# Research Notes: CheXNet & DenseNet-121

**Project:** AI Radiology Assistant  
**Author:** Malek Hayouni  
**Date:** 2026

Notes summarizing the key papers and design decisions behind this project's model architecture.

---

## 1. CheXNet: Key Paper

**Title:** CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning  
**Authors:** Rajpurkar et al., Stanford ML Group  
**Published:** 2017  https://arxiv.org/abs/1711.05225

### What it does
CheXNet is a 121-layer convolutional neural network trained on the NIH Chest X-ray14 dataset.
It outputs a probability for each of 14 pathology classes simultaneously (multi-label classification).

### Key results
| Pathology | CheXNet AUC | Radiologist F1 |
|---|---|---|
| Pneumonia | **0.768** | 0.387 |
| Atelectasis | 0.8094 | — |
| Cardiomegaly | 0.9248 | — |
| Effusion | 0.8638 | — |
| Overall avg | ~0.841 | — |

The model **surpassed radiologist-level performance on pneumonia** detection.

### Architecture decision
- Base: DenseNet-121 pretrained on ImageNet
- Final layer: replaced with `Linear(1024 → 14)` + `Sigmoid()`
- Loss: Binary Cross-Entropy (BCE)  each label treated independently
- Optimizer: Adam, lr=0.001, decayed on plateau
- Input: 224×224, normalized with ImageNet mean/std

---

## 2. DenseNet-121 Architecture

**Title:** Densely Connected Convolutional Networks  
**Authors:** Huang et al., 2017 — https://arxiv.org/abs/1608.06993

### Why DenseNet works well for medical imaging

In a standard CNN, each layer only receives input from the previous layer.  
In DenseNet, **each layer receives feature maps from ALL preceding layers**:

```
Standard CNN:   x0 → x1 → x2 → x3
DenseNet:       x0 → x1 → x2 → x3
                      ↗       ↗
                x0 ──────────
```

This means:
- **Feature reuse** early low-level features (edges, textures) are preserved all the way to the output
- **Fewer parameters** than ResNet for equivalent depth
- **Stronger gradients** shorter paths for backpropagation
- Particularly useful for detecting subtle findings in X-rays (nodules, early infiltration)

### DenseNet-121 structure
| Block | Layers |
|---|---|
| Initial conv | 7×7, stride 2, 64 filters |
| Dense block 1 | 6 layers |
| Transition 1 | 1×1 conv + 2×2 avg pool |
| Dense block 2 | 12 layers |
| Transition 2 | 1×1 conv + 2×2 avg pool |
| Dense block 3 | 24 layers |
| Transition 3 | 1×1 conv + 2×2 avg pool |
| Dense block 4 | 16 layers |
| Global avg pool | → 1024-dim vector |
| Classifier | 14-unit sigmoid (CheXNet) |

---

## 3. Grad-CAM Explainability

**Title:** Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization  
**Authors:** Selvaraju et al., 2017 https://arxiv.org/abs/1610.02391

### Why it matters here
A radiology AI that outputs a probability with no spatial context is not clinically useful.
Grad-CAM generates a **heatmap showing which regions of the X-ray most influenced the prediction**,
giving the radiologist a visual explanation alongside the score.

### How it works
1. Run a forward pass get prediction for class `c`
2. Backpropagate the gradient of class `c` score w.r.t. the last convolutional feature maps
3. Global-average-pool the gradients → importance weights per channel
4. Weighted sum of feature maps → raw CAM
5. Apply ReLU (keep only positive activations)
6. Upsample to input image size (224×224)
7. Overlay with jet colormap on original X-ray

### Target layer
For DenseNet-121: `model.densenet.features.denseblock4`  
This is the final dense block  (strongest).

---

## 4. Training Strategy (planned)

### Loss function
```python
# Weighted BCE to handle class imbalance
pos_weight = (N - n_pos) / n_pos  # per class
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
```

### Optimizer
```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.1
)
```

### Evaluation metric
**AUC-ROC per class** (standard for this dataset, used in CheXNet paper)  
Target: ≥ 0.80 AUC on the NIH official test split

### Data augmentation
```python
transforms.RandomHorizontalFlip(),
transforms.RandomRotation(10),
transforms.ColorJitter(brightness=0.2, contrast=0.2),
```
Note: no vertical flip (chest X-rays have fixed orientation)

---

## 5. Alternatives Considered

| Model | Params | CXR14 AUC | Notes |
|---|---|---|---|
| DenseNet-121 (CheXNet) | 7M | ~0.841 |  Chosen  well benchmarked ✅|
| ResNet-50 | 25M | ~0.820 | Larger weaker feature reuse |
| EfficientNet-B4 | 19M | ~0.850 | Slightly better, less documented for CXR |
| Vision Transformer (ViT-B) | 86M | ~0.860 | Too large for portfolio scope |

**Decision:** DenseNet-121 chosen for direct comparability with the CheXNet paper and availability of pretrained weights.

---

## 6. References

1. Rajpurkar et al. (2017) — CheXNet — https://arxiv.org/abs/1711.05225
2. Huang et al. (2017) — DenseNet — https://arxiv.org/abs/1608.06993
3. Selvaraju et al. (2017) — Grad-CAM — https://arxiv.org/abs/1610.02391
4. Wang et al. (2017) — NIH Chest X-ray14 — https://arxiv.org/abs/1705.02315
5. torchvision DenseNet — https://pytorch.org/vision/stable/models/densenet.html